In [ ]:
import wandb

# Replace YOUR_API_KEY with your actual key (as a string)
wandb.login(key='4bf1b0217052027d110bec5a39e55d5a41679e59')


In [ ]:
# ---------------------------------------------------------------------
# 0) IMPORTS
# ---------------------------------------------------------------------
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, roc_curve, auc, precision_recall_curve
from tensorflow.keras.callbacks import Callback
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint
from kerastuner.tuners import RandomSearch
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam


In [ ]:
# ---------------------------------------------------------------------
# 1) READ CSV AND SET PATHS
# ---------------------------------------------------------------------
df = pd.read_csv("/kaggle/input/aerial-cactus/train.csv")
image_dir = "/kaggle/input/aerial-cactus/train/train"
test_dir = "/kaggle/input/aerial-cactus/test/test"
image_size = (32, 32)

In [ ]:
# ---------------------------------------------------------------------
# 2) LOAD IMAGES INTO MEMORY
# ---------------------------------------------------------------------
def load_images(df, img_dir, target_size):
    images = []
    labels = []

    for _, row in df.iterrows():
        img_path = os.path.join(img_dir, row['id'])
        img = cv2.imread(img_path)
        img = cv2.resize(img, target_size)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img / 255.0  # normalize
        images.append(img)
        labels.append(row['has_cactus'])

    return np.array(images), np.array(labels)

In [ ]:
# ---------------------------------------------------------------------
# 3) SPLIT DATA AND LOAD
# ---------------------------------------------------------------------
train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df['has_cactus'], 
    random_state=42
)

X_train, y_train = load_images(train_df, image_dir, image_size)
X_val, y_val = load_images(val_df, image_dir, image_size)

In [ ]:
# ---------------------------------------------------------------------
# 4) BUILD MODEL
# ---------------------------------------------------------------------

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(32, 32, 3)),
    MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

'''
#def build_model(hp):
#    model = Sequential()
#    model.add(Conv2D(
#        hp.Int('conv_1_filters', min_value=32, max_value=128, step=32),
#        (3, 3), activation='relu', input_shape=(32, 32, 3)))
#    model.add(MaxPooling2D((2,2)))
    
#    model.add(Conv2D(
#        hp.Int('conv_2_filters', min_value=32, max_value=128, step=32),
#        (3, 3), activation='relu'))
#    model.add(MaxPooling2D((2,2)))
    
#    model.add(Flatten())
#    model.add(Dense(
#        hp.Int('dense_units', min_value=32, max_value=256, step=32),
#        activation='relu'))
#    model.add(Dropout(hp.Float('dropout', 0.2, 0.5, step=0.1)))
#    model.add(Dense(1, activation='sigmoid'))

#    model.compile(
#        optimizer=Adam(learning_rate=hp.Choice('lr', [1e-2, 1e-3, 1e-4])),
#        loss='binary_crossentropy',
#        metrics=['accuracy']
#    )
    
#    return model
'''
'''
#model = Sequential([
#    Conv2D(32, (5,5), activation='relu', input_shape=(32, 32, 3)),
#    MaxPooling2D((2,2)),
#    Conv2D(100, (5,5), activation='relu'),
#    MaxPooling2D((2,2)),
#    Flatten(),
#    Dense(1600, activation='relu'),
#    Dense(500, activation='relu'),
#    Dense(1, activation='sigmoid')  # binary output
#])
'''

wandb.init(project="cactus-classifier", name="batch-size-128")

model.compile(
    optimizer=Adam(),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
class MetricsCallback(Callback):
    def __init__(self, val_data):
        super().__init__()
        self.X_val, self.y_val = val_data
        self.val_f1s = []
        self.val_precisions = []
        self.val_recalls = []

    def on_epoch_end(self, epoch, logs=None):
        y_pred_probs = self.model.predict(self.X_val, verbose=0)
        y_pred = (y_pred_probs > 0.5).astype(int).flatten()
        y_true = self.y_val

        precision = precision_score(y_true, y_pred)
        recall = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)

        self.val_precisions.append(precision)
        self.val_recalls.append(recall)
        self.val_f1s.append(f1)

        print(f" Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")



metrics_cb = MetricsCallback(val_data=(X_val, y_val))

In [ ]:
'''
tuner = RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory='tuner_dir',
    project_name='cactus_classifier'
)

tuner.search(X_train, y_train, epochs=10, validation_data=(X_val, y_val))
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()
'''

In [ ]:
# ---------------------------------------------------------------------
# 5) TRAIN MODEL
# ---------------------------------------------------------------------
EPOCHS = 50
#history = model.fit(
#    X_train, y_train,
#    validation_data=(X_val, y_val),
#    batch_size=64,
#    epochs=EPOCHS,
#    verbose=1
#)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=128,
    epochs=EPOCHS,
    callbacks=[
        WandbMetricsLogger(),         # Logs all training/val metrics
        WandbModelCheckpoint("model.keras") # Optional: logs model checkpoints
    ],
    verbose=1
)

In [ ]:
y_probs = model.predict(X_val)

# Step 2: Convert probabilities to binary labels
y_pred = (y_probs > 0.5).astype(int).flatten()

y_true = y_val

#The dataset is unbalances, with approx 75% images with a cactus and 25% without. Thus compute the following metrics to asses performance on each class
precision = precision_score(y_true, y_pred) #For images with a cactus
recall = recall_score(y_true, y_pred) #For images without a cactus
f1 = f1_score(y_true, y_pred) #Metric that summarizes precision and recall

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

cm = confusion_matrix(y_val, y_pred)
ConfusionMatrixDisplay(cm).plot()

In [ ]:
fpr, tpr, thresholds = roc_curve(y_val, y_probs)  # y_probs = model.predict(X_val)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC)")
plt.legend()
plt.show()

In [ ]:
classified_indices = np.where(y_pred == y_val)[0]
misclassified_indices = np.where(y_pred != y_val)[0]

for i in classified_indices[5:7]:  
    plt.imshow(X_val[i])
    plt.title(f"True: {y_val[i]}, Predicted: {y_pred[i]}")
    plt.show()

for i in misclassified_indices[5:7]: 
    plt.imshow(X_val[i])
    plt.title(f"True: {y_val[i]}, Predicted: {y_pred[i]}")
    plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_val, y_probs)
plt.plot(recall, precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.xlim(0.95, 1)
plt.ylim(0.95, 1)
plt.grid()
plt.show()

In [ ]:
# ---------------------------------------------------------------------
# 6) PLOT METRICS
# ---------------------------------------------------------------------
plt.figure()
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.figure()
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

plt.figure()
plt.plot(metrics_cb.val_recalls, label='Val Recall')
plt.plot(metrics_cb.val_precisions, label='Val Precision')
plt.plot(metrics_cb.val_f1s, label='Val F1 Score')
plt.title('Validation Metrics Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# ---------------------------------------------------------------------
# 7) PREDICT TEST DATA
# ---------------------------------------------------------------------
'''
test_filenames = sorted(os.listdir(test_dir))

def load_test_images(filenames, img_dir, target_size):
    images = []
    for fname in filenames:
        img_path = os.path.join(img_dir, fname)
        img = cv2.imread(img_path)
        img = cv2.resize(img, target_size)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img / 255.0
        images.append(img)
    return np.array(images)

X_test = load_test_images(test_filenames, test_dir, image_size)

preds = model.predict(X_test)
pred_labels = (preds > 0.5).astype(int).reshape(-1)
'''

In [ ]:
# ---------------------------------------------------------------------
# 8) SAVE SUBMISSION
# ---------------------------------------------------------------------
'''
submission_df = pd.DataFrame({
    "id": test_filenames,
    "has_cactus": pred_labels
})
submission_df.to_csv("my_submission.csv", index=False)
print("Submission saved as my_submission.csv")
'''